# Integration of OpenCellID Infrastructure Features

This notebook extends Dataset19 by integrating cellular-infrastructure information obtained from OpenCellID.

OpenCellID records are cleaned, converted into spatial points, and assigned to the labelled 1 km × 1 km grid cells. Six grid-level infrastructure predictors are created and combined with the environmental and socioeconomic predictors from Dataset19.

The resulting infrastructure-enhanced dataset is referred to as Dataset20.

※Related dissertation sections
 - Section 2.6 (Public Geospatial Data as Predictors)
 - Section 3.1 (Research Approach—Data Understanding and Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 1. Load Dataset19
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

REPOSITORY_ROOT = Path(
    "/content/drive/MyDrive/england-lte-5g-signal-prediction"
)
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from config import BUILD_DIR, OPENCELLID_FILE

base_dir = str(BUILD_DIR)

dataset19_path = os.path.join(
    base_dir,
    "08_Final_Datasets",
    "dataset19_s1_s2_worldcover_population_nightlight.gpkg"
)

dataset19 = gpd.read_file(
    dataset19_path,
    layer="dataset19_environmental_features"
)

print("Dataset19 shape:", dataset19.shape)
print("Unique grid IDs:", dataset19["grid_id"].nunique())
print("CRS:", dataset19.crs)

print("\nColumns:")
print(dataset19.columns.tolist())

display(dataset19.head())

## 1. Load and Inspect the Input Data

Dataset19, containing the signal labels and 11 environmental and socioeconomic predictors, is loaded as the baseline input.

The raw OpenCellID file is then inspected for its structure, radio-technology distribution, operator identifiers, missing coordinates, and coordinate ranges.

※Related dissertation sections
 - Section 3.1 (Research Approach—Data Understanding)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 2. Inspect OpenCellID Raw Data
# ============================================================

opencellid_path = str(OPENCELLID_FILE)

# Validate source file
if not os.path.exists(opencellid_path):
    raise FileNotFoundError(
        f"OpenCellID source file not found: {opencellid_path}"
    )

# Read the first five rows
ocid_sample = pd.read_csv(
    opencellid_path,
    nrows=5,
    low_memory=False
)

print("OpenCellID file:", opencellid_path)
print("File exists:", os.path.exists(opencellid_path))
print("Sample shape:", ocid_sample.shape)

print("\nColumns:")
print(ocid_sample.columns.tolist())

print("\nFirst five rows:")
display(ocid_sample)

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 3. Load and Inspect Full OpenCellID Data
# ============================================================

ocid_columns = [
    "radio",
    "mcc",
    "net",
    "area",
    "cell",
    "unit",
    "lon",
    "lat",
    "range",
    "samples",
    "updated"
]

ocid = pd.read_csv(
    opencellid_path,
    usecols=ocid_columns,
    low_memory=False
)

print("OpenCellID shape:", ocid.shape)

print("\nRadio technology distribution:")
print(ocid["radio"].value_counts(dropna=False))

print("\nUnique operators (net):")
print(ocid["net"].nunique(dropna=True))

print("\nMissing coordinates:")
print(ocid[["lon", "lat"]].isna().sum())

print("\nCoordinate ranges:")
print(ocid[["lon", "lat"]].agg(["min", "max"]))

display(ocid.head())

## 2. Spatial Preparation and Integration

OpenCellID longitude and latitude records are converted from WGS84 (EPSG:4326) to the British National Grid (EPSG:27700).

Duplicate cellular records are removed using their network identifiers and coordinates. The remaining records are spatially assigned to the labelled 1 km grid cells using a point-in-polygon join.

※ Related dissertation sections
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 4. Prepare Grid and OpenCellID Geometries
# ============================================================

# Dataset19 already contains valid EPSG:27700 geometry
grid_gdf = dataset19.copy()

# Convert OpenCellID records to point geometries
ocid_gdf = gpd.GeoDataFrame(
    ocid.copy(),
    geometry=gpd.points_from_xy(
        ocid["lon"],
        ocid["lat"]
    ),
    crs="EPSG:4326"
).to_crs(epsg=27700)

# Validate geometries
print("Dataset19 grids:", f"{len(grid_gdf):,}")
print("Unique grid IDs:", grid_gdf["grid_id"].nunique())
print("Grid CRS:", grid_gdf.crs)
print("Invalid grid geometries:", (~grid_gdf.geometry.is_valid).sum())

print("\nOpenCellID records:", f"{len(ocid_gdf):,}")
print("OpenCellID CRS:", ocid_gdf.crs)
print("Missing point geometries:", ocid_gdf.geometry.isna().sum())
print("Invalid point geometries:", (~ocid_gdf.geometry.is_valid).sum())

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 5. Clean and Spatially Join OpenCellID Records
# ============================================================

# Remove duplicate cellular records using network identifiers
ocid_clean = ocid_gdf.drop_duplicates(
    subset=[
        "radio", "mcc", "net",
        "area", "cell", "unit",
        "lon", "lat"
    ]
).copy()

print("Records before deduplication:", f"{len(ocid_gdf):,}")
print("Records after deduplication :", f"{len(ocid_clean):,}")
print("Duplicates removed          :", f"{len(ocid_gdf) - len(ocid_clean):,}")

# Spatially join records to labelled 1 km grids
ocid_joined = gpd.sjoin(
    ocid_clean[
        ["radio", "net", "geometry"]
    ],
    grid_gdf[
        ["grid_id", "geometry"]
    ],
    how="inner",
    predicate="within"
)

print("\nOpenCellID records joined:", f"{len(ocid_joined):,}")
print(
    "Grids containing OpenCellID records:",
    f"{ocid_joined['grid_id'].nunique():,}"
)

display(ocid_joined.head())

## 3. Construction of Cellular-Infrastructure Features

The spatially matched OpenCellID records are aggregated by `grid_id` to create six infrastructure predictors:

- 'cell_count': total number of OpenCellID records;
- 'operator_count': number of unique network operators;
- 'gsm_count': number of GSM records;
- 'umts_count': number of UMTS records;
- 'lte_count': number of LTE records; and
- 'nr_count': number of 5G NR records.

The technology-specific counts are validated against the total record count for each grid.

※ Related dissertation sections
 - Section 2.6 (Public Geospatial Data as Predictors)
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction)
 - Appendix C (Predictor Definitions).

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 6. Create Grid-Level Infrastructure Features
# ============================================================

# Total OpenCellID records per grid
cell_count = (
    ocid_joined
    .groupby("grid_id")
    .size()
    .rename("cell_count")
)

# Number of unique operators per grid
operator_count = (
    ocid_joined
    .groupby("grid_id")["net"]
    .nunique()
    .rename("operator_count")
)

# Technology-specific record counts
tech_counts = (
    ocid_joined
    .groupby(["grid_id", "radio"])
    .size()
    .unstack(fill_value=0)
)

for tech in ["GSM", "UMTS", "LTE", "NR"]:
    if tech not in tech_counts.columns:
        tech_counts[tech] = 0

tech_counts = tech_counts[
    ["GSM", "UMTS", "LTE", "NR"]
].rename(
    columns={
        "GSM": "gsm_count",
        "UMTS": "umts_count",
        "LTE": "lte_count",
        "NR": "nr_count"
    }
)

# Combine infrastructure features
infrastructure_features = pd.concat(
    [
        cell_count,
        operator_count,
        tech_counts
    ],
    axis=1
).reset_index()

# Validate total count against technology counts
tech_total = infrastructure_features[
    ["gsm_count", "umts_count", "lte_count", "nr_count"]
].sum(axis=1)

if not tech_total.equals(infrastructure_features["cell_count"]):
    raise ValueError("Technology counts do not match total cell counts.")

print("Infrastructure feature shape:", infrastructure_features.shape)
print("Unique grid IDs:", infrastructure_features["grid_id"].nunique())

print("\nColumns:")
print(infrastructure_features.columns.tolist())

display(infrastructure_features.head())

## 4. Creation and Export of Dataset20

The six OpenCellID infrastructure predictors are merged with Dataset19 using 'grid_id'.

Grid cells without matched OpenCellID records are assigned zero values. The completed Dataset20 contains the original 11 environmental and socioeconomic predictors together with the six infrastructure predictors.

The final dataset is validated and exported in CSV and GeoPackage formats.

※ Related dissertation sections
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction)
 - Appendix C (Reproducibility and Predictor Definitions).

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 7. Merge Infrastructure Features with Dataset19
# ============================================================

infra_cols = [
    "cell_count",
    "operator_count",
    "gsm_count",
    "umts_count",
    "lte_count",
    "nr_count"
]

# Merge infrastructure features
dataset20 = dataset19.merge(
    infrastructure_features,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

# Assign zero to grids without OpenCellID records
dataset20[infra_cols] = (
    dataset20[infra_cols]
    .fillna(0)
    .astype("int32")
)

# Validate merge
if len(dataset20) != len(dataset19):
    raise ValueError("The number of rows changed during merging.")

if dataset20["grid_id"].nunique() != len(dataset20):
    raise ValueError("Duplicate grid IDs were found in Dataset20.")

print("Dataset20 shape:", dataset20.shape)
print("Unique grid IDs:", dataset20["grid_id"].nunique())

print("\nMissing infrastructure values:")
print(dataset20[infra_cols].isna().sum())

print("\nInfrastructure summary:")
display(dataset20[infra_cols].describe())

display(dataset20.head())

In [ ]:
# ============================================================
# Dataset20 (OpenCellID)
# Cell 8. Save Final Dataset20
# ============================================================

dataset20_output_dir = os.path.join(
    base_dir,
    "08_Final_Datasets"
)

csv_path = os.path.join(
    dataset20_output_dir,
    "dataset20_final.csv"
)

gpkg_path = os.path.join(
    dataset20_output_dir,
    "dataset20_final.gpkg"
)

# Save tabular dataset without geometry
dataset20.drop(
    columns="geometry"
).to_csv(
    csv_path,
    index=False
)

# Save spatial dataset with geometry
dataset20.to_file(
    gpkg_path,
    layer="dataset20_final",
    driver="GPKG"
)

print("Final Dataset20 shape:", dataset20.shape)
print("Unique grid IDs:", dataset20["grid_id"].nunique())

print("\nCSV saved:")
print(csv_path)

print("\nGeoPackage saved:")
print(gpkg_path)

print("\nInfrastructure dtypes:")
print(dataset20[infra_cols].dtypes)